Multi agent system using an actual LLM

In [25]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph_supervisor import create_supervisor
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph

Only actions that an LLM needs to execute are tools. The other agents are not tools, they are separate graphs (or nodes).

In [26]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [27]:
@tool
def web_search(query: str) -> str:
    """Search the web."""
    return f""""
    LangGraph is an orchestration framework for building AI agents.
    It supports stateful workflows, tools, memory and multi-agent systems.
    
    Search Query: {query}"""

In [28]:
research_agent = create_agent(
    llm,
    tools=[web_search],
    name="research_agent",
    system_prompt="""
            You are a researcher.
            Only collect facts.
            Never write articles."""
)

In [29]:
writer_agent = create_agent(
    llm,
    tools=[],
    name="writer_agent",
    system_prompt="""
You are a technical writer.

Use only the provided research. If no related information, mention "Information not given"

Write a concise summary.
"""
)

In [30]:
workflow = create_supervisor(
    [research_agent, writer_agent],
    model=llm,
    prompt="""
You supervise two agents.

Research Agent:
- Collects facts.

Writer Agent:
- Writes the final response.

Always research first.
Then send to writer.
"""
)

In [31]:
app = workflow.compile(
    checkpointer=InMemorySaver()
)

In [37]:
config={
    "configurable": {
        "thread_id": "thread-1"
    }
}

In [38]:
result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Research Pytorch and write a summary."
            }
        ]
    }, config=config
)

In [39]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Research Pytorch and write a summary.
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_agent (fc_2a0a7ea2-2c01-4252-9a88-24b3e0ea24f2)
 Call ID: fc_2a0a7ea2-2c01-4252-9a88-24b3e0ea24f2
  Args:
    : {}
================================= Tool Message =================================
Name: transfer_to_research_agent

Successfully transferred to research_agent
================================== Ai Message ==================================
Name: research_agent
================================== Ai Message ==================================
Name: research_agent

Transferring back to supervisor
Tool Calls:
  transfer_back_to_supervisor (6e1853df-0f9e-458c-81bf-58073e388f46)
 Call ID: 6e1853df-0f9e-458c-81bf-58073e388f46
  Args:
================================= Tool Message =================================
Name: transfer_back_

In [36]:
for msg in result["messages"]:
    print(msg)

content='Research LangGraph and write a summary.' additional_kwargs={} response_metadata={} id='a62efa64-4bbe-45bb-b031-e1c736c59f9a'
content='' additional_kwargs={'reasoning_content': 'We need to supervise two agents. First, ask research_agent to collect facts about LangGraph. Then send to writer_agent to produce summary. Use transfer_to_research_agent, then transfer_to_writer_agent.', 'tool_calls': [{'id': 'fc_e819d52a-c984-4612-873b-076454aab505', 'function': {'arguments': '{"task":"Collect facts about LangGraph, including its purpose, features, architecture, use cases, and any notable projects or libraries."}', 'name': 'transfer_to_research_agent'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 185, 'total_tokens': 279, 'completion_time': 0.199252763, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.008804181, 'prompt_tokens_details': None, 'queue_time': 0.285537314, 'total_time': 0.208056944}, 'model_name'